# Persistency Risk Prediction Model

This notebook builds a machine learning model to predict whether a policy is likely to lapse (`lapse_flag`).

The implementation follows the feature contract and preprocessing rules defined in the Persistency Model Plan.

# Import Required Libraries

This project uses:

- Pandas and NumPy for data manipulation.
- Matplotlib and Seaborn for visualization.
- Scikit-learn for preprocessing, model training, and evaluation.

These libraries provide the tools required to build the Persistency Risk Prediction Model.

In [1]:
# Data Handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data Splitting
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer
)
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

# Metrics
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    classification_report,
    confusion_matrix
)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold, cross_val_score
# Add this under # Data Splitting
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.base import clone

# Load the Dataset

The training dataset contains policyholder information, coverage details, payment preferences, affordability ratios, and the persistency target variable (`lapse_flag`).

The dataset will be loaded into a Pandas DataFrame for further analysis and preprocessing.

In [2]:
df = pd.read_csv("backend\\data\\persistency_training_data.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (50000, 29)


,policy_id,age,annual_income,dependents,existing_life_coverage,outstanding_loans,desired_coverage,recommended_coverage,product_id,product_name,...,years_since_issue,mortality_score,base_rate_per_thousand,premium_amount,premium_income_ratio,coverage_income_ratio,lapse_probability,persistency_risk_score,persistency_risk_category,lapse_flag
0,POL0000001,43,815000,0,0,2520000,9300000,10670000,LIC-WL-213,Enduring Security Whole Life - Series E,...,19,25.78,18.4,65182.41,0.0800,2.9202,0.0824,8.24,Low,1
1,POL0000002,18,641100,0,0,70000,7320000,6480000,LIC-U-160,Bluechip Multiplier ULIP,...,0,1.00,36.0,22190.84,0.0346,1.2167,0.1706,17.06,Low,1
2,POL0000003,48,394700,1,1530000,490000,6160000,2910000,LIC-MB-191,Cash Flow Life Plan (Series B),...,10,40.63,48.0,20210.50,0.0512,0.6081,0.0901,9.01,Low,0
3,POL0000004,57,453000,3,0,440000,6210000,4970000,LIC-WL-197,Heritage Wealth & Life - Series C,...,15,37.06,15.2,45976.93,0.1015,3.4658,0.0916,9.16,Low,0
4,POL0000005,34,563600,1,0,0,5180000,5640000,LIC-TL-187,Cancer Shield (Series F) Term,...,6,28.42,5.0,37102.48,0.0658,10.0071,0.1268,12.68,Low,0


# Initial Data Exploration

Before training a machine learning model, it is important to understand:

- Dataset dimensions
- Data types
- Missing values
- Target variable distribution

This helps identify potential data quality issues and ensures that preprocessing steps are applied correctly.

In [3]:
print(df.info())

print("\nMissing Values:\n")
print(df.isnull().sum())

print("\nTarget Distribution:\n")
print(df["lapse_flag"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   policy_id                  50000 non-null  str    
 1   age                        50000 non-null  int64  
 2   annual_income              50000 non-null  int64  
 3   dependents                 50000 non-null  int64  
 4   existing_life_coverage     50000 non-null  int64  
 5   outstanding_loans          50000 non-null  int64  
 6   desired_coverage           50000 non-null  int64  
 7   recommended_coverage       50000 non-null  int64  
 8   product_id                 50000 non-null  str    
 9   product_name               50000 non-null  str    
 10  product_type               50000 non-null  str    
 11  product_risk_profile       50000 non-null  str    
 12  coverage_amount            50000 non-null  int64  
 13  coverage_adequacy_ratio    50000 non-null  float64
 14  p

# Feature Engineering

Create the preference-match indicators required by the model plan.

The Persistency Model Plan specifies two derived features:

1. `payment_mode_matches_preference`
2. `autopay_matches_preference`

These features capture whether a customer's actual payment setup matches their stated preference, which may influence policy persistency.

In [4]:
df["payment_mode_matches_preference"] = (
    df["payment_mode"] ==
    df["payment_mode_preference"]
).astype(int)

df["autopay_matches_preference"] = (
    df["autopay_enabled"] ==
    df["autopay_preference"]
).astype(int)

# Feature Selection

Only approved Version 1 features defined in the Persistency Model Plan are included in model training.

The selected features are grouped into:

- Numerical Features
- Categorical Features
- Binary Features
- Derived Features

Excluded columns are intentionally omitted to prevent data leakage and model overfitting.

In [13]:
numerical_features = [
    "age",
    "annual_income",
    "dependents",
    "coverage_amount",
    "premium_amount",
    "policy_term",
    "premium_income_ratio",
    "coverage_income_ratio",
    "coverage_adequacy_ratio",
    "years_since_issue"
]

categorical_features = [
    "product_type",
    "payment_mode",
    "payment_mode_preference"
]

binary_features = [
    "autopay_enabled",
    "autopay_preference",
    "payment_mode_matches_preference",
    "autopay_matches_preference"
]

target = "lapse_flag"

selected_features = (
    numerical_features +
    categorical_features +
    binary_features
)

X = df[selected_features]
y = df[target]

# Data Splitting

The dataset is divided using a stratified sampling strategy to preserve the original class distribution.

Split configuration:

- Training Set: 70%
- Validation Set: 15%
- Test Set: 15%
- Random Seed: 42

The test set will remain untouched until final model evaluation.

In [14]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (35000, 17)
Validation: (7500, 17)
Test: (7500, 17)


# Data Preprocessing Pipeline

The preprocessing pipeline follows the Persistency Model Plan.

### Numerical Features
- Missing values are imputed using the median.
- Monetary variables are log-transformed using `log1p()`.
- Numerical features are standardized using StandardScaler.

### Categorical Features
- Missing values are imputed using the most frequent category.
- One-Hot Encoding is applied.
- Unknown categories are handled automatically.

### Binary Features
- Binary variables are passed directly without additional transformation.

All preprocessing steps are integrated into the machine learning pipeline to prevent data leakage.

In [15]:
monetary_features = [
    "annual_income",
    "coverage_amount",
    "premium_amount"
]

other_numeric_features = [
    "age",
    "dependents",
    "policy_term",
    "premium_income_ratio",
    "coverage_income_ratio",
    "coverage_adequacy_ratio",
    "years_since_issue"
]

monetary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p)),
    ("scaler", StandardScaler())
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("money", monetary_pipeline, monetary_features),
    ("numeric", numeric_pipeline, other_numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
    ("binary", "passthrough", binary_features)
])

# Stratified K-Fold Cross-Validation

To ensure robust model evaluation and avoid overfitting on a single validation split, we implement Stratified K-Fold Cross-Validation. This technique splits the training dataset into $K$ folds while preserving the percentage of samples for each class (`lapse_flag`).

We will evaluate the primary performance metrics requested by the model plan—specifically **PR-AUC (Average Precision)**, **Brier Score**, and **Log Loss**—across all folds.

In [23]:
# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_with_cross_validation(pipeline, X_train_data, y_train_data):
    """
    Performs Stratified 5-Fold Cross-Validation and prints out-of-fold metrics.
    """
    pr_aucs = []
    brier_scores = []
    log_losses = []
    roc_aucs = []

    # Reset index to avoid matching issues with pandas series inside the loop
    X_tr = X_train_data.reset_index(drop=True)
    y_tr = y_train_data.reset_index(drop=True)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_tr, y_tr), 1):
        # Clone the pipeline to ensure a fresh model instance for each fold
        fold_pipeline = clone(pipeline)

        # Split the data
        X_fold_train, X_fold_val = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
        y_fold_train, y_fold_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]

        # Fit the model on the fold training data
        fold_pipeline.fit(X_fold_train, y_fold_train)

        # Predict probabilities for the fold validation data
        y_prob = fold_pipeline.predict_proba(X_fold_val)[:, 1]

        # Calculate primary metrics
        roc_auc = roc_auc_score(y_fold_val, y_prob)
        pr_auc = average_precision_score(y_fold_val, y_prob)
        brier = brier_score_loss(y_fold_val, y_prob)
        loss = log_loss(y_fold_val, y_prob)

        roc_aucs.append(roc_auc)
        pr_aucs.append(pr_auc)
        brier_scores.append(brier)
        log_losses.append(loss)

        print(f"Fold {fold} -> ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f} | Brier Score: {brier:.4f} | Log Loss: {loss:.4f}")

    print("\n" + "="*50)
    print(f"Mean ROC-AUC    : {np.mean(roc_aucs):.4f} (+/- {np.std(roc_aucs):.4f})")
    print(f"Mean PR-AUC     : {np.mean(pr_aucs):.4f} (+/- {np.std(pr_aucs):.4f})")
    print(f"Mean Brier Score: {np.mean(brier_scores):.4f} (+/- {np.std(brier_scores):.4f})")
    print(f"Mean Log Loss   : {np.mean(log_losses):.4f} (+/- {np.std(log_losses):.4f})")
    print("="*50)

# Model 1: Logistic Regression

Logistic Regression serves as the baseline model.

Advantages:
- Fast training
- Easy interpretation
- Produces calibrated probability estimates

This model establishes a benchmark against which more complex models can be compared.

In [24]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Run Stratified Cross-Validation
print("Evaluating Logistic Regression with 5-Fold Stratified CV...")
evaluate_with_cross_validation(logistic_model, X_train, y_train)

# Fit on full training set after cross-validation
logistic_model.fit(X_train, y_train)

Evaluating Logistic Regression with 5-Fold Stratified CV...
Fold 1 -> ROC-AUC: 0.6629 | PR-AUC: 0.2745 | Brier Score: 0.1202 | Log Loss: 0.3983
Fold 2 -> ROC-AUC: 0.6852 | PR-AUC: 0.3045 | Brier Score: 0.1179 | Log Loss: 0.3904
Fold 3 -> ROC-AUC: 0.6714 | PR-AUC: 0.3032 | Brier Score: 0.1183 | Log Loss: 0.3934
Fold 4 -> ROC-AUC: 0.6544 | PR-AUC: 0.2603 | Brier Score: 0.1212 | Log Loss: 0.4012
Fold 5 -> ROC-AUC: 0.6813 | PR-AUC: 0.2980 | Brier Score: 0.1184 | Log Loss: 0.3920

Mean ROC-AUC    : 0.6710 (+/- 0.0114)
Mean PR-AUC     : 0.2881 (+/- 0.0176)
Mean Brier Score: 0.1192 (+/- 0.0013)
Mean Log Loss   : 0.3951 (+/- 0.0041)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['age','annual_income','dependents',...,'autopay_preference', 'payment_mode_matches_preference','autopay_matches_preference']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('money', ...), ('numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (

# Model 2: Random Forest

Random Forest is an ensemble learning method that combines multiple decision trees.

Advantages:
- Captures nonlinear relationships
- Handles feature interactions
- Reduces overfitting through bagging

This model serves as a nonlinear comparison to the Logistic Regression baseline.

In [25]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['age','annual_income','dependents',...,'autopay_preference', 'payment_mode_matches_preference','autopay_matches_preference']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('money', ...), ('numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (

In [22]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42))
])

# Run Stratified Cross-Validation
print("Evaluating Random Forest model with 5-Fold Stratified CV...")
evaluate_with_cross_validation(rf_model, X_train, y_train)

# Fit on full training set after cross-validation
rf_model.fit(X_train, y_train)

Evaluating Logistic Regression with 5-Fold Stratified CV...
Fold 1 -> PR-AUC: 0.2550 | Brier Score: 0.1226 | Log Loss: 0.4079
Fold 2 -> PR-AUC: 0.2712 | Brier Score: 0.1212 | Log Loss: 0.4028
Fold 3 -> PR-AUC: 0.2801 | Brier Score: 0.1210 | Log Loss: 0.4040
Fold 4 -> PR-AUC: 0.2366 | Brier Score: 0.1248 | Log Loss: 0.4144
Fold 5 -> PR-AUC: 0.2650 | Brier Score: 0.1215 | Log Loss: 0.4037

Mean ROC-AUC    : 0.6424 (+/- 0.0108)
Mean PR-AUC     : 0.2616 (+/- 0.0150)
Mean Brier Score: 0.1222 (+/- 0.0014)
Mean Log Loss   : 0.4066 (+/- 0.0043)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['age','annual_income','dependents',...,'autopay_preference', 'payment_mode_matches_preference','autopay_matches_preference']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('money', ...), ('numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (

# Model 3: Gradient Boosting

Gradient Boosting is the primary nonlinear candidate identified in the Persistency Model Plan.

Advantages:
- Learns complex patterns
- Often achieves strong predictive performance
- Sequentially improves weak learners

This model is expected to provide the strongest predictive capability among the candidate models.

In [26]:
gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['age','annual_income','dependents',...,'autopay_preference', 'payment_mode_matches_preference','autopay_matches_preference']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('money', ...), ('numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (

In [28]:
gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

# Run Stratified Cross-Validation
print("Evaluating Gradient boosting model with 5-Fold Stratified CV...")
evaluate_with_cross_validation(gb_model, X_train, y_train)

# Fit on full training set after cross-validation
gb_model.fit(X_train, y_train)

Evaluating Gradient boosting model with 5-Fold Stratified CV...
Fold 1 -> ROC-AUC: 0.6648 | PR-AUC: 0.2675 | Brier Score: 0.1206 | Log Loss: 0.3984
Fold 2 -> ROC-AUC: 0.6766 | PR-AUC: 0.2918 | Brier Score: 0.1189 | Log Loss: 0.3936
Fold 3 -> ROC-AUC: 0.6748 | PR-AUC: 0.3070 | Brier Score: 0.1180 | Log Loss: 0.3922
Fold 4 -> ROC-AUC: 0.6502 | PR-AUC: 0.2566 | Brier Score: 0.1217 | Log Loss: 0.4024
Fold 5 -> ROC-AUC: 0.6740 | PR-AUC: 0.2828 | Brier Score: 0.1194 | Log Loss: 0.3950

Mean ROC-AUC    : 0.6681 (+/- 0.0098)
Mean PR-AUC     : 0.2811 (+/- 0.0177)
Mean Brier Score: 0.1197 (+/- 0.0013)
Mean Log Loss   : 0.3963 (+/- 0.0037)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['age','annual_income','dependents',...,'autopay_preference', 'payment_mode_matches_preference','autopay_matches_preference']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('money', ...), ('numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (

# Model Evaluation

The candidate models are evaluated on the validation dataset.

### Primary Metrics

- Precision-Recall AUC (PR-AUC)
- Brier Score
- Log Loss

### Supporting Metrics

- ROC-AUC
- Precision
- Recall
- F1 Score

The model with the strongest validation performance will be selected for final testing.

In [29]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model
}

for name, model in models.items():

    probs = model.predict_proba(X_val)[:, 1]

    print("\n" + "="*50)
    print(name)
    print("="*50)

    print(
        "PR-AUC:",
        round(average_precision_score(y_val, probs), 4)
    )

    print(
        "ROC-AUC:",
        round(roc_auc_score(y_val, probs), 4)
    )

    print(
        "Brier Score:",
        round(brier_score_loss(y_val, probs), 4)
    )

    print(
        "Log Loss:",
        round(log_loss(y_val, probs), 4)
    )


Logistic Regression
PR-AUC: 0.305
ROC-AUC: 0.6897
Brier Score: 0.1178
Log Loss: 0.3896

Random Forest
PR-AUC: 0.2691
ROC-AUC: 0.6639
Brier Score: 0.1208
Log Loss: 0.4

Gradient Boosting
PR-AUC: 0.3087
ROC-AUC: 0.7001
Brier Score: 0.1174
Log Loss: 0.3877


# Save Model Artifacts

The final approved model and supporting metadata are saved for deployment and future inference.

Artifacts generated:

- model.joblib
- feature_schema.json
- metadata.json
- metrics.json
- risk_thresholds.json

In [30]:
X_final = df[selected_features]
y_final = df[target]

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

final_model.fit(X_final, y_final)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['age','annual_income','dependents',...,'autopay_preference', 'payment_mode_matches_preference','autopay_matches_preference']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('money', ...), ('numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (

In [33]:
import joblib
import json
from pathlib import Path

# Create directories if they don't exist
models_dir = Path("backend\\models")
artifacts_dir = Path("backend\\artifacts\\persistency_model")

models_dir.mkdir(exist_ok=True)
artifacts_dir.mkdir(exist_ok=True)

# --------------------------------------------------
# Saving the model
# --------------------------------------------------

joblib.dump(
    final_model,
    models_dir / "persistency_prediction_model_v1.joblib"
)

['backend\\models\\persistency_prediction_model_v1.joblib']

# Code Cell- Feature Schema Json

In [34]:
import json

feature_schema = {
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "binary_features": binary_features,
    "target": "lapse_flag"
}

with open( artifacts_dir / "feature_schema.json","w") as f:
    json.dump(feature_schema, f, indent=4)

print("feature_schema.json saved.")

feature_schema.json saved.


# Code Cell- Meta DataJson

In [35]:
from datetime import datetime
import sklearn

metadata = {
    "model_version": "v1.0",
    "random_seed": 42,
    "training_timestamp": str(datetime.now()),
    "dataset_rows": int(df.shape[0]),
    "dataset_columns": int(df.shape[1]),
    "sklearn_version": sklearn.__version__
}

with open( artifacts_dir / "metadata.json","w") as f:
    json.dump(metadata, f, indent=4)

print("metadata.json saved.")

metadata.json saved.


# Code Cell- Metrics Jason

In [37]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss
)

test_probs = final_model.predict_proba(X_final)[:, 1]

metrics = {
    "pr_auc": float(
        average_precision_score(
            y_final,
            test_probs
        )
    ),
    "roc_auc": float(
        roc_auc_score(
            y_final,
            test_probs
        )
    ),
    "brier_score": float(
        brier_score_loss(
            y_final,
            test_probs
        )
    ),
    "log_loss": float(
        log_loss(
            y_final,
            test_probs
        )
    )
}

with open( artifacts_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("metrics.json saved.")

metrics.json saved.


# Code Cell- Risk Tresholds Json

In [38]:
risk_thresholds = {
    "classification_threshold": 0.50,
    "risk_categories": {
        "Low": [0, 25],
        "Medium": [26, 50],
        "High": [51, 75],
        "Severe": [76, 100]
    }
}

with open( artifacts_dir / "risk_thresholds.json", "w") as f:
    json.dump(risk_thresholds, f, indent=4)

print("risk_thresholds.json saved.")

risk_thresholds.json saved.


# Final Verification Cell

In [46]:
from pathlib import Path

artifacts_dir = Path("backend/artifacts/persistency_model")

print("Generated Artifacts:\n")

for file in artifacts_dir.iterdir():
    print(file.name)

Generated Artifacts:

feature_schema.json
metadata.json
metrics.json
risk_thresholds.json


In [47]:
final_probs = final_model.predict_proba(X_final)[:, 1]

print("PR-AUC:", average_precision_score(y_final, final_probs))

print("ROC-AUC:", roc_auc_score(y_final, final_probs))

print("Brier score:", brier_score_loss(y_final,final_probs))

PR-AUC: 0.3265170363045929
ROC-AUC: 0.702613304574913
Brier score: 0.11618411689285327


# Final Model Evaluation

The selected model is evaluated on the untouched test dataset to estimate its performance on unseen data.